In [ ]:
# ==============================================================================
# INSTALACIÓN DE DEPENDENCIAS COMPATIBLES
# ==============================================================================
#
# Se fijan versiones compatibles de las dependencias utilizadas por ClimaX para
# garantizar reproducibilidad del entorno experimental. El reinicio del runtime
# fuerza la recarga de las bibliotecas instaladas.
# ==============================================================================

print("[INFO] Instalando dependencias compatibles...")

!pip -q uninstall -y numpy pandas timm climax

!pip -q install numpy==1.26.4
!pip -q install pandas==2.2.2
!pip -q install timm==0.6.13
!pip -q install git+https://github.com/microsoft/ClimaX.git

print("[OK] Instalación terminada.")

import os

os.kill(
    os.getpid(),
    9,
)

[INFO] Instalando dependencias compatibles...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bokeh 3.8.2 requires pandas>=1.2, which is not installed.
geopandas 1.1.4 requires pandas>=2.0.0, which is not installed.
inequality 1.1.2 requires pandas>=2.1, which is not installed.
xarray 2025.12.0 requires pandas>=2.2, which is not installed.
dask-cuda 26.2.0 requires pandas>=1.3, which is not installed.
access 1.1.10.post3 requires pandas>=2.1.0, which is not installed.
yfinance 0.2.66 requires pandas>=1.3.0, which is not installed.
gradio 6.20.0 requires pandas<4.0,>=1.0, which is not installed.
tobler 0.14.0 requires pandas>=2.2, which is not installed.
statsmodels 0.14.6 requires pandas!=2.1.0,>=1.4, whi

In [37]:
# ==============================================================================
# COMPATIBILIDAD LEGACY NUMPY
# ==============================================================================
#
# ClimaX y algunas dependencias internas utilizan alias eliminados en versiones
# recientes de NumPy. Se restauran temporalmente para mantener compatibilidad
# sin modificar el código fuente externo.
# ==============================================================================

import numpy as np

np.float = float
np.int = int
np.bool = bool
np.object = object
np.complex = complex

print("[OK] Compatibilidad NumPy aplicada.")

[OK] Compatibilidad NumPy aplicada.


In [2]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS Y DEPENDENCIAS
# ==============================================================================
#
# Las dependencias se centralizan después del parche NumPy para asegurar que
# ClimaX cargue correctamente bajo el entorno compatible definido previamente.
# ==============================================================================

import os

import numpy as np
import pandas as pd
import torch
from climax.arch import ClimaX
import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive

print("[INFO] Montando Google Drive...")

drive.mount(
    "/content/drive",
)

print("[OK] Google Drive montado.")

[INFO] Montando Google Drive...
Mounted at /content/drive
[OK] Google Drive montado.


In [38]:
# ==============================================================================
# CONFIGURACIÓN DEL EXPERIMENTO
# ==============================================================================
#
# La parametrización del escenario permite reutilizar el mismo flujo de
# inferencia para diferentes horizontes temporales sin modificar la lógica
# principal del modelo.
# ==============================================================================

# ==============================================================================
# CONFIGURACIÓN DEL PROYECTO (Rutas Parametrizables)
# ==============================================================================
# Si cambias de entorno (Local vs Colab) o renombras la carpeta,
# solo necesitas modificar esta única variable.

BASE_PATH = "/content/drive/My Drive/Experimentos/ClimaXGanador"

SCENARIO = "medio_plazo"

SCENARIOS = {
    "corto_plazo": {
        "seq_len": 24,
        "pred_len": 3,
        "stride": 3,
        "alpha": 0.95,  # Peso dominante para el modelo Entrenado
    },
    "medio_plazo": {
        "seq_len": 96,
        "pred_len": 48,
        "stride": 12,
        "alpha": 0.85,  # Equilibrio para el crecimiento inicial del error
    },
    "largo_plazo": {
        "seq_len": 168,
        "pred_len": 72,
        "stride": 24,
        "alpha": 0.50,  # Regresión 50/50 hacia la climatología
    },
}

config = SCENARIOS[SCENARIO]

SEQ_LEN = config["seq_len"]
PRED_LEN = config["pred_len"]
STRIDE = config["stride"]
ALPHA = config["alpha"]

print(f"[INFO] Escenario: {SCENARIO}")

print(f"[INFO] Lookback: {SEQ_LEN} horas")

print(f"[INFO] Forecast: {PRED_LEN} horas")

print(f"[INFO] Stride: {STRIDE} horas")

print(f"[INFO] Alpha RAF: {ALPHA}")

[INFO] Escenario: medio_plazo
[INFO] Lookback: 96 horas
[INFO] Forecast: 48 horas
[INFO] Stride: 12 horas
[INFO] Alpha RAF: 0.85


In [39]:
# ==============================================================================
# CARGA DE MEMORY BANK RAF
# ==============================================================================
#
# El banco de memoria contiene representaciones latentes generadas previamente
# mediante ClimaX. La normalización L2 permite utilizar producto punto como
# aproximación eficiente de similitud coseno durante el retrieval.
# ==============================================================================

RAF_PATH = os.path.join(
    BASE_PATH,
    "data",
    "climax_multiscenario_tensors",
    SCENARIO,
)

EMB_PATH = os.path.join(
    RAF_PATH,
    "embeddings.pt",
)

IDX_PATH = os.path.join(
    RAF_PATH,
    "indices.pt",
)

memory_bank = torch.load(
    EMB_PATH,
)

memory_bank = memory_bank / (
    memory_bank.norm(
        dim=1,
        keepdim=True,
    )
    + 1e-8
)

meta = torch.load(
    IDX_PATH,
)

print("[INFO] RAF cargado.")

print(f"[INFO] Embeddings: {memory_bank.shape}")

print(f"[INFO] Metadata: {meta}")

[INFO] RAF cargado.
[INFO] Embeddings: torch.Size([11680, 1024])
[INFO] Metadata: {'scenario': 'medio_plazo', 'seq_len': 96, 'pred_len': 48, 'stride': 12, 'num_samples': 11680}


In [40]:
# ==============================================================================
# DEFINICIÓN DEL MODELO CLIMAX PARA INFERENCIA
# ==============================================================================
#
# Wrapper utilizado para cargar el modelo ClimaX fine-tuned y generar
# predicciones multihorizonte. El backbone conserva la representación
# espacial aprendida durante el fine-tuning, mientras que el adaptador
# temporal y el decoder reconstruyen la variable objetivo temperatura.
# ==============================================================================
# from climax.arch import ClimaX
# importar de nuevo si da un error de importación


class ClimaXFineTuningWrapper(nn.Module):

    def __init__(
        self,
        default_vars,
        img_size=(4, 4),
        patch_size=2,
        seq_len=SEQ_LEN,
        pred_len=PRED_LEN,
    ):

        super().__init__()

        self.default_vars = default_vars
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.img_size = img_size

        # ------------------------------------------------------------------
        # Backbone ClimaX fine-tuned
        # Debe coincidir con el modelo guardado
        # ------------------------------------------------------------------

        self.backbone = ClimaX(
            default_vars=default_vars,
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=1024,
            depth=8,
            num_heads=16,
        )

        # ------------------------------------------------------------------
        # Adaptación temporal:
        # [seq_len] -> [pred_len]
        # ------------------------------------------------------------------

        self.temporal_adapter = nn.Linear(seq_len, pred_len)

        # ------------------------------------------------------------------
        # Decoder temperatura
        # ------------------------------------------------------------------

        self.decoder = nn.Sequential(nn.Linear(1024, 256), nn.GELU(), nn.Linear(256, 1))

    def forward(self, x):

        batch_size = x.shape[0]

        # Entrada:
        # [B,T,C,H,W]
        #
        # ClimaX recibe:
        # [B*T,C,H,W]

        x = x.reshape(batch_size * self.seq_len, x.shape[2], x.shape[3], x.shape[4])

        lead_times = torch.zeros(batch_size * self.seq_len, device=x.device, dtype=torch.float32)

        # Encoder espacial ClimaX
        features = self.backbone.forward_encoder(x, lead_times, self.default_vars)

        # Resultado esperado:
        # [B*T, patches, 1024]
        #
        # Ejemplo:
        # [B*T,4,1024]

        features = features.view(batch_size, self.seq_len, features.shape[1], features.shape[2])

        # Cambiar dimensión temporal al final:
        #
        # [B,T,P,1024]
        # ->
        # [B,P,1024,T]

        features = features.permute(0, 2, 3, 1)

        # Proyección temporal:
        #
        # [seq_len] -> [pred_len]

        features = self.temporal_adapter(features)

        # Restaurar orden:
        #
        # [B,P,1024,pred]
        # ->
        # [B,pred,P,1024]

        features = features.permute(0, 3, 1, 2)

        # Decoder por patch

        output = self.decoder(features)

        output = output.squeeze(-1)

        # Actualmente:
        # [B,pred,4]
        #
        # Los 4 patches representan:
        # una grilla 2x2

        output = output.view(batch_size, self.pred_len, 2, 2)

        # Interpolación:
        # 2x2 -> 4x4

        output = F.interpolate(
            output.reshape(batch_size * self.pred_len, 1, 2, 2),
            size=self.img_size,
            mode="bilinear",
            align_corners=False,
        )

        # Salida final:
        #
        # [B,pred_len,1,4,4]

        output = output.view(batch_size, self.pred_len, 1, self.img_size[0], self.img_size[1])

        return output

In [41]:
# ==============================================================================
# CARGA DEL MODELO FINE-TUNED
# ==============================================================================
#
# Se cargan los pesos obtenidos durante el entrenamiento de ClimaX para el
# escenario seleccionado. La arquitectura debe coincidir exactamente con la
# utilizada durante el fine-tuning.
# ==============================================================================
variables_meteorologicas = [
    "temperaturaMedia",
    "humedadRelativa",
    "presionBarometrica",
    "precipitacion",
    "radiacionSolar",
    "viento_u",
    "viento_v",
]

MODEL_PATH = os.path.join(BASE_PATH, "models", "climax_finetuned", SCENARIO, "climax_best.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Modelo:", MODEL_PATH)

model = ClimaXFineTuningWrapper(
    default_vars=variables_meteorologicas, seq_len=SEQ_LEN, pred_len=PRED_LEN
)

checkpoint = torch.load(MODEL_PATH, map_location=device)

model.load_state_dict(checkpoint)

model = model.to(device)

model.eval()

print("Modelo ClimaX cargado correctamente")

🚀 Device: cuda
📦 Modelo: /content/drive/My Drive/Experimentos/final/models/climax_finetuned/medio_plazo/climax_best.pt
✅ Modelo ClimaX cargado correctamente


In [42]:
# ==============================================================================
# EXTRACCIÓN DE EMBEDDINGS PARA RETRIEVAL
# ==============================================================================
#
# Se obtiene una representación latente de la ventana meteorológica actual
# utilizando únicamente el encoder ClimaX. Esta representación será utilizada
# para buscar ventanas históricas similares dentro del memory bank RAF.
# ==============================================================================


@torch.no_grad()
def get_embedding(x):

    # Entrada:
    # x = [B,T,C,H,W]

    x = x.to(device)

    B, T, C, H, W = x.shape

    # ClimaX procesa cada instante temporal
    # individualmente

    x = x.reshape(B * T, C, H, W)

    lead_times = torch.zeros(B * T, device=device, dtype=torch.float32)

    features = model.backbone.forward_encoder(x, lead_times, model.default_vars)

    # Restaurar dimensión temporal

    features = features.view(B, T, features.shape[1], features.shape[2])

    # Pooling temporal + espacial
    #
    # [B,T,P,D]
    # ->
    # [B,D]

    embedding = features.mean(dim=(1, 2))

    return embedding.cpu()

In [43]:
# ==============================================================================
# RETRIEVAL TOP-K
# ==============================================================================
#
# Se compara el embedding generado por la ventana meteorológica actual contra
# la base RAF almacenada mediante similitud coseno. Las ventanas históricas con
# mayor similitud representan patrones climáticos análogos.
# ==============================================================================


def retrieve_topk(query, k=5):
    # NUEVO: Asegurar que el query esté en el mismo dispositivo que el memory_bank (GPU/CUDA)
    query = query.to(memory_bank.device)

    # Normalización del embedding consulta
    query = query / (query.norm(dim=1, keepdim=True) + 1e-8)

    # Similitud coseno:
    # memory_bank -> [N,D]
    # query       -> [B,D]
    similarity = torch.matmul(memory_bank, query.T)

    similarity = similarity.squeeze()

    # Selección de vecinos más similares
    indices = torch.topk(similarity, k=k).indices

    return indices

In [44]:
# ==============================================================================
# CONTEXTO RAF
# ==============================================================================
#
# A partir de los vecinos históricos recuperados se construye una representación
# contextual mediante el promedio de sus embeddings. Este contexto representa
# patrones meteorológicos similares encontrados en la memoria histórica.
# ==============================================================================


def RAF_step(x):

    # Generar embedding de la ventana actual

    query_embedding = get_embedding(x)

    # Recuperar patrones similares

    indices = retrieve_topk(query_embedding, k=5)

    # Obtener contexto histórico

    context = memory_bank[indices].mean(dim=0)

    return context

In [45]:
# ==============================================================================
# PREDICCIÓN AUTOREGRESIVA CON RAF
# ==============================================================================
#
# Se realiza la predicción iterativa utilizando la ventana temporal actual.
# En cada paso se genera una predicción con ClimaX y posteriormente se actualiza
# la ventana de entrada desplazando el historial e incorporando el nuevo
# horizonte generado.
# ==============================================================================


@torch.no_grad()
def predict_raf(x_init, steps=1):
    x = x_init.clone()
    predictions = []

    for step_idx in range(steps):
        # 1. Obtener el embedding de la ventana actual
        query_embedding = get_embedding(x)

        # 2. Recuperar los índices de los K vecinos más similares
        indices = retrieve_topk(query_embedding, k=5)

        # 3. Obtener las temperaturas reales (targets) históricas
        retrieved_targets = Y_historical[indices.cpu()]  # Shape: [5, pred_len, 1, 4, 4]
        analog_forecast = retrieved_targets.mean(dim=0).unsqueeze(
            0
        )  # Shape: [1, pred_len, 1, 4, 4]

        # 4. Predicción base del modelo ClimaX
        pred_climax = model(x.to(device)).cpu()  # Shape: [1, pred_len, 1, 4, 4]

        # 5. Combinar ClimaX con el pronóstico análogo de RAF
        alpha = ALPHA
        pred_final = alpha * pred_climax + (1 - alpha) * analog_forecast

        predictions.append(pred_final)

        # 6. Actualización temporal de la ventana (solo si hay más pasos por predecir)
        if step_idx < steps - 1:
            # Repetimos el último valor conocido de los otros 6 canales (persistencia)
            # x[:, -1:, 1:] extrae los otros 6 canales en el último paso de tiempo conocido
            last_known_others = x[:, -1:, 1:].repeat(
                1, PRED_LEN, 1, 1, 1
            )  # Shape: [1, PRED_LEN, 6, 4, 4]

            # Concatenamos la temperatura predicha (canal 0) con los otros 6 canales
            new_step = torch.cat(
                [pred_final, last_known_others], dim=2
            )  # Shape: [1, PRED_LEN, 7, 4, 4]

            # Deslizamos la ventana temporal
            x = torch.cat([x[:, PRED_LEN:], new_step], dim=1)

    return torch.cat(predictions, dim=1)

In [46]:
# ==============================================================================
# INFERENCIA SOBRE DATASET DE PRUEBA
# ==============================================================================
#
# Se generan predicciones para todas las ventanas del conjunto de prueba.
# La salida conserva la estructura espacial de la grilla meteorológica:
#
# [muestras, horizonte, variable, alto, ancho]
#
# ==============================================================================


@torch.no_grad()
def forecast_full_dataset(X_test, model, device):
    model.eval()
    predictions = []

    for i in range(len(X_test)):
        # Pasamos x_init sin mandarlo a GPU directo para que predict_raf lo maneje
        x_init = X_test[i : i + 1]

        # NUEVO: Usar la función predict_raf con RAF
        y_hat = predict_raf(x_init, steps=1)

        predictions.append(y_hat.cpu())

    return torch.cat(predictions, dim=0)

**Ejecucion**

In [47]:
# ==============================================================================
# CARGA DEL DATASET DE PRUEBA
# ==============================================================================
#
# Se carga el conjunto de prueba generado durante la preparación de tensores.
# Además de las variables de entrada y salida, se recupera la información
# necesaria para reconstruir las predicciones en las estaciones meteorológicas
# originales.
# ==============================================================================
# ==============================================================================
# CARGA DEL DATASET DE PRUEBA Y BASE HISTÓRICA (TRAIN + VAL)
# ==============================================================================
test_path = os.path.join(RAF_PATH, "climax_test.pt")
test_data = torch.load(test_path, map_location="cpu")

X_test = test_data["X"]
y_test = test_data["Y"]
stations_map = test_data["stations_map"]
mean = test_data["target_mean"]
std = test_data["target_scale"]

# Cargar targets de entrenamiento
train_path = os.path.join(RAF_PATH, "climax_train.pt")
train_data = torch.load(train_path, map_location="cpu")
Y_train = train_data["Y"]

# Intentar cargar targets de validación para completar la base histórica de RAF
val_path = os.path.join(RAF_PATH, "climax_val.pt")
if os.path.exists(val_path):
    val_data = torch.load(val_path, map_location="cpu")
    Y_val = val_data["Y"]
    # Concatenamos ambos para coincidir con las 46,744 muestras del memory_bank
    Y_historical = torch.cat([Y_train, Y_val], dim=0)
    print(
        f"[OK] Base histórica construida: Train ({Y_train.shape[0]}) + Val ({Y_val.shape[0]}) = {Y_historical.shape[0]} muestras"
    )
else:
    Y_historical = Y_train
    print(
        f"[WARNING] No se encontró climax_val.pt. Usando solo Train ({Y_historical.shape[0]} muestras)"
    )

print("X_test:", X_test.shape)
print("Y_train (histórico):", Y_train.shape)
print("y_test:", y_test.shape)

print("stations_map:")
print(stations_map)

print("mean:", mean.shape)
print("std :", std.shape)

[OK] Base histórica construida: Train (10220) + Val (1460) = 11680 muestras
X_test: torch.Size([1638, 96, 7, 4, 4])
Y_train (histórico): torch.Size([10220, 48, 1, 4, 4])
y_test: torch.Size([1638, 48, 1, 4, 4])
stations_map:
{'Belisario': (0, 0), 'Carapungo': (0, 1), 'Cotocollao': (0, 2), 'ElCamal': (0, 3), 'LosChillos': (1, 0), 'Tumbaco': (1, 1)}
mean: torch.Size([6])
std : torch.Size([6])


In [48]:
# ==============================================================================
# GENERACIÓN DE PREDICCIONES
# ==============================================================================
#
# Se mueve el modelo y la memoria RAF a GPU para acelerar la inferencia.
# Posteriormente se generan las predicciones sobre todas las ventanas del
# conjunto de prueba.
# ==============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model.eval()

memory_bank = memory_bank.to(device)

forecast = forecast_full_dataset(X_test, model, device)

print("Forecast shape:", forecast.shape)

Forecast shape: torch.Size([1638, 48, 1, 4, 4])


In [49]:
# ==============================================================================
# DESNORMALIZACIÓN DE TEMPERATURA
# ==============================================================================
#
# Se transforma la salida normalizada del modelo nuevamente a unidades reales
# de temperatura utilizando la media y desviación estándar calculadas durante
# la construcción del dataset.
# ==============================================================================

# Variable objetivo:
# temperaturaMedia -> índice 0

mean_temp = float(mean[0])

std_temp = float(std[0])

Y_pred = forecast.numpy()

Y_true = y_test.numpy()

# Desnormalización

Y_pred = (Y_pred * std_temp) + mean_temp

Y_true = (Y_true * std_temp) + mean_temp

print("Pred shape:", Y_pred.shape)

print("True shape:", Y_true.shape)

print("Mean:", mean_temp, "Std:", std_temp)

Pred shape: (1638, 48, 1, 4, 4)
True shape: (1638, 48, 1, 4, 4)
Mean: 14.11220645904541 Std: 3.1652138233184814


In [50]:
# ==============================================================================
# RECONSTRUCCIÓN TEMPORAL DE PREDICCIONES
# ==============================================================================
#
# Las predicciones espaciales son transformadas nuevamente a registros
# tabulares por estación meteorológica. Cada ventana mantiene su fecha inicial
# considerando el stride utilizado durante la generación de tensores.
#
# ==============================================================================

start_date = pd.Timestamp("2024-01-01 00:00:00")

rows = []

for i in range(Y_pred.shape[0]):

    # Fecha inicial real de la ventana

    window_start = start_date + pd.Timedelta(hours=i * STRIDE)

    for h in range(PRED_LEN):

        # Horizonte de predicción
        #
        # inicia en 0:
        # h=0 -> primera hora predicha

        fecha = window_start + pd.Timedelta(hours=h)

        for estacion, (r, c) in stations_map.items():

            rows.append([fecha, h, estacion, float(Y_pred[i, h, 0, r, c])])

df_pred = pd.DataFrame(rows, columns=["fechaHora", "horizon", "estacion", "pred_temp"])

print(df_pred.head())

print("Inicio:", df_pred["fechaHora"].min())

print("Fin:", df_pred["fechaHora"].max())

   fechaHora  horizon    estacion  pred_temp
0 2024-01-01        0   Belisario  12.428060
1 2024-01-01        0   Carapungo  12.615433
2 2024-01-01        0  Cotocollao  13.038567
3 2024-01-01        0     ElCamal  13.243927
4 2024-01-01        0  LosChillos  12.972546
Inicio: 2024-01-01 00:00:00
Fin: 2026-03-31 11:00:00


In [51]:
# ==============================================================================
# CARGA DE DATOS OBSERVADOS
# ==============================================================================
#
# Se recuperan las temperaturas reales desde el feature store original y se
# filtran únicamente las fechas correspondientes al periodo de prueba.
# ==============================================================================

df_raw = pd.read_parquet("/content/drive/My Drive/Experimentos/ClimaXGanador/data/mdt_feature_store_2008.parquet")

df_test = (
    df_raw[df_raw["fechaHora"] >= "2024-01-01"]
    .sort_values(["fechaHora", "estacion"])
    .reset_index(drop=True)
)

print(df_test.head())

   fechaHora    estacion       fecha  hora  humedadRelativa  precipitacion  \
0 2024-01-01   Belisario  2024-01-01     0        92.720001       0.000000   
1 2024-01-01   Carapungo  2024-01-01     0        97.489998       0.000000   
2 2024-01-01  Cotocollao  2024-01-01     0        93.949997       0.000000   
3 2024-01-01     ElCamal  2024-01-01     0        88.292198       0.059859   
4 2024-01-01  LosChillos  2024-01-01     0        93.389999       0.000000   

   presionBarometrica  radiacionSolar  temperaturaMedia  mes  ...  viento_v  \
0          727.770020             0.0         11.570000    1  ... -0.026516   
1          746.020020             6.9         13.270000    1  ...  1.090650   
2          732.729980             0.0         12.560000    1  ...  0.187285   
3          727.765991             0.0         12.182379    1  ...  0.123294   
4          761.719971             0.0         13.450000    1  ... -0.189698   

   precipitacion_log  hora_sin  hora_cos  mes_sin   mes_

In [52]:
# ==============================================================================
# UNIÓN PREDICCIÓN VS REALIDAD
# ==============================================================================
#
# Se combinan las predicciones generadas por ClimaX con los valores observados
# de temperatura para cada estación y horizonte temporal.
# ==============================================================================

df_real = df_test[["fechaHora", "estacion", "temperaturaMedia"]].rename(
    columns={"temperaturaMedia": "real_temp"}
)

df_final = df_pred.merge(df_real, on=["fechaHora", "estacion"], how="left")[
    ["fechaHora", "horizon", "estacion", "real_temp", "pred_temp"]
]

print(df_final.head())

   fechaHora  horizon    estacion  real_temp  pred_temp
0 2024-01-01        0   Belisario  11.570000  12.428060
1 2024-01-01        0   Carapungo  13.270000  12.615433
2 2024-01-01        0  Cotocollao  12.560000  13.038567
3 2024-01-01        0     ElCamal  12.182379  13.243927
4 2024-01-01        0  LosChillos  13.450000  12.972546


In [53]:
# ==============================================================================
# EXPORTACIÓN DE RESULTADOS
# ==============================================================================
#
# Se almacenan las predicciones finales en formato parquet para facilitar el
# análisis posterior, cálculo de métricas y generación de gráficos.
# El nombre del archivo mantiene el escenario evaluado.
# ==============================================================================

EXPORT_PATH = os.path.join(BASE_PATH, "results", f"predictions_{SCENARIO}.parquet")

os.makedirs(os.path.dirname(EXPORT_PATH), exist_ok=True)

df_final.to_parquet(EXPORT_PATH, index=False)

print("Resultados guardados en:")

print(EXPORT_PATH)

Resultados guardados en:
/content/drive/My Drive/Experimentos/final/results/predictions_medio_plazo.parquet
